# Working With datasets

This tutorial is a sequel to [Tutorial 01](https://lc.llnl.gov/jupyter/user/cdoutrix/notebooks/git/Kosh/examples/Example_01_Add_Data_To_Datasets.ipynb) which should have been successfully ran before this tutotrial.

In this tutorial we will open a store, look for some datasets of interest, search for failed nodes and time, and mark the datasets as failed if necessary.


## Connect to store (using sina local file)


In [1]:
from  kosh import KoshStore
import os

# local tutorial sql file
kosh_example_sql_file = "kosh_example.sql"

# connect to store
store = KoshStore(engine="sina", username=os.environ["USER"], sql='sql', db_path=kosh_example_sql_file)

## Looping through datasets

Let's look for our "IBM project"-related datsets

In [2]:
datasets = store.search(project="IBM")
print("We identified {} possible datasets".format(len(datasets)))

We identified 320 possible datasets


## Working with datasets and files.

Now we are going to identify failed nodes and their earliest time of failure.


In [12]:
import numpy
import h5py
import json
from dask.distributed import Lock

def identify_bad_nodes(filename,
                       metrics=["metrics_0", "metrics_1", "metrics_2", "metrics_3", "metrics_4", "metrics_5"],
                       jsonfile="bad_nodes.json",
                       jsonkey=None,
                       threshold=0., verbose=False):
    out = {}
    if os.path.exists(jsonfile):
        with open(jsonfile) as f:
            existing = json.load(f)
    else:
        existing = {}
    if jsonkey is None:
        jsonkey = filename
    if jsonkey in existing:
        return existing[jsonkey]
    for i, metric in enumerate(metrics):
        if verbose:
            print("\tMetric:", metric)
        h5pointer = h5py.File(filename, mode="r")["node"][metric]
        num_cycles = h5pointer.shape[0]
        # eliminate first 3rd of run
        # start = num_cycles - 2*num_cycles//3
        # or look at all cycles
        start = 0
        if verbose:
            print("\t\t", h5pointer.shape, start)
        data = h5pointer[start:]
        if verbose:
            print("\t\t", data.shape, data.min(), data.max())
        bad = numpy.argwhere(data<=threshold)
        out[metric] = []
        if bad.shape[0] == 0:
            print("\t\tNo bad nodes! Yeah for success")
        else:
            bad_nodes = list(set(bad[:,1]))
            if verbose:
                print("\t\tFor",metric, "There are", len(bad_nodes), "actual bad nodes", bad.shape, bad[0], bad[-1])
                print("\t\tTotal bad_nodes was", bad.shape[0], "which is ", bad.shape[0]/data.size*100., "%")
            for b in bad_nodes:
                indices = numpy.argwhere(bad[:,1]==b)
                node = h5pointer[:,b]
                if node.max() != 0:
                    out[metric].append([b, h5pointer.shape[0], numpy.take(bad[:,0], indices)[:,0].tolist()])
    try:  # if we're using dask
        l = Lock("json")
        l.acquire()
    except:
        pass
    existing[jsonkey] = out
    with open(jsonfile,"w") as f:
        json.dump(existing, f)
    try:
        # Release the dask lock
        l.release()
    except:
        pass
    return out

In [13]:
# Keeping only datsets for which hdf5 is available
use_dask = True
if use_dask:
    from dask.distributed import Client
    client = Client()


bad_nodes = {}
for ds in datasets:
    print("looking at:", ds.name, ds.__associated_data__)
    hdf5 = ds.search(type="hdf5")
    if len(hdf5)>0:
        #print("Found an hdf5 file")
        h5 = hdf5[0]
        if use_dask:
            try:
                a = client.submit(identify_bad_nodes, h5.uri, jsonkey=ds.name)
                bad_nodes[ds.name] = a
            except Exception as err:
                print(err)
                pass
        else:
            try:
                bad_nodes[ds.name] = identify_bad_nodes(h5.uri, jsonkey=ds.name, verbose=False)
            except:
                pass

/g/g19/cdoutrix/miniconda3/envs/kosh/lib/python3.7/site-packages/distributed/dashboard/core.py:72: UserWarning: 
Port 8787 is already in use. 
Perhaps you already have a cluster running?
Hosting the diagnostics dashboard on a random port instead.
  warnings.warn("\n" + msg)


looking at: molar.0.0_shock.1.3_taper.0.8_skew.0.6 ['42d21f42fb3e11e99e1c001e67a135ba', '42e4f7cafb3e11e99e1c001e67a135ba']
Using i loader: <kosh.sina.core.KoshSinaFileLoader object at 0x2aaab4e902e8>
looking at: molar.0.0_shock.1.1_taper.0.4_skew.0.6 ['40f5ef1efb3e11e99e1c001e67a135ba']
looking at: molar.1_shock.1.1_taper.0.2_skew.0.8 ['48a9f728fb3e11e99e1c001e67a135ba', '48bec90afb3e11e99e1c001e67a135ba']
Using i loader: <kosh.sina.core.KoshSinaFileLoader object at 0x2aaab4e902e8>
looking at: molar.1_shock.1.1_taper.0.6_skew.0.4 ['4dc72f46fb3e11e99e1c001e67a135ba', '4ddacc0efb3e11e99e1c001e67a135ba']
Using i loader: <kosh.sina.core.KoshSinaFileLoader object at 0x2aaab4e902e8>
looking at: molar.1_shock.1.7_taper.0.4_skew.0.8 ['422b8c40fb3e11e99e1c001e67a135ba']
looking at: molar.0.0_shock.1.5_taper.0.6_skew.0.4 ['3b890ba6fb3e11e99e1c001e67a135ba', '3b9bc48afb3e11e99e1c001e67a135ba']
Using i loader: <kosh.sina.core.KoshSinaFileLoader object at 0x2aaab4e902e8>
looking at: molar.0.0_shoc

In [ ]:
import json
for k in bad_nodes: 
    print("K:",k)
    try:
        if os.path.exists("bad.json"):
            bad = json.load(open("bad.json"))
        else:
            bad = {}
        if not k in bad:
            bad[k] = bad_nodes[k].result()
            with open("bad.json","w") as f:
                json.dump(bad, f)
        print("done")
    except:
        pass

K: molar.0.0_shock.1.3_taper.0.8_skew.0.6


In [6]:
error = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.1_shock.1.7_taper.0.6_skew.0.8/slurm-340974.out'
no_error = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.1_shock.1.3_taper.0.6_skew.0.6/slurm-340933.out'
error_goal_reached = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.0.3_shock.1.3_taper.0.8_skew.0.9/slurm-340778.out'

In [7]:
#h5 = store.search(name=error_goal_reached.split("/")[-2])[0]
#h5 = h5.search(type="hdf5")[0].uri
#print("Using:", h5)
#bad = identify_bad_nodes(h5)

In [11]:
bad_nodes['molar.0.0_shock.1.3_taper.0.8_skew.0.6'].result()

AttributeError: module 'posixpath' has no attribute 'exist'